In [3]:
import gzip
import shutil

input_path = "/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz"

output_path = "/kaggle/working/accepted_2007_to_2018Q4.csv"

with gzip.open(input_path, "rb") as f_in:
    with open(output_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("File extracted successfully!")

File extracted successfully!


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
df=pd.read_csv("/kaggle/working/accepted_2007_to_2018Q4.csv", low_memory=False)

In [6]:
print(df.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

In [7]:
print(df.shape) 
print(len(df.columns))  

(2260701, 151)
151


In [8]:
pd.set_option('display.max_rows', None)

nulls = df.isna().sum()
null_df = pd.DataFrame({
    'null_count': nulls,
    'null_pct': (nulls / len(df) * 100).round(2)
}).sort_values('null_pct', ascending=False)

print("Columns with nulls:", (nulls > 0).sum(), "out of", df.shape[1])
print("Columns with more than 50% nulls:", (null_df['null_pct'] > 50).sum())

null_df

Columns with nulls: 150 out of 151
Columns with more than 50% nulls: 44


,null_count,null_pct
member_id,2260701,100.00
orig_projected_additional_accrued_interest,2252050,99.62
hardship_reason,2249784,99.52
hardship_payoff_balance_amount,2249784,99.52
hardship_last_payment_amount,2249784,99.52
payment_plan_start_date,2249784,99.52
hardship_type,2249784,99.52
hardship_status,2249784,99.52
hardship_start_date,2249784,99.52
deferral_term,2249784,99.52


In [9]:
keep = ["mths_since_last_delinq", "mths_since_last_record",
        "mths_since_last_major_derog", "mths_since_recent_bc_dlq",
        "mths_since_recent_revol_delinq"]

null_pct = df.isna().mean() * 100
cols_to_drop = [c for c in null_pct[null_pct >= 50].index if c not in keep]
df = df.drop(columns=cols_to_drop)
df = df.dropna(subset=["loan_status"]) 
print("Dropped columns:", len(cols_to_drop))  
print("Remaining columns:", df.shape[1])      

Dropped columns: 39
Remaining columns: 112


In [10]:
leakage = [
    "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low",
    "debt_settlement_flag",
]
df = df.drop(columns=leakage)

In [11]:
print(df["emp_title"].nunique(), df["title"].nunique(), df["zip_code"].nunique())

512694 63154 956


In [12]:
drop_cols = ["emp_title", "title","url"]
df = df.drop(columns=drop_cols)
print(df.shape[1])   

94


In [13]:
closed_status = ["Fully Paid", "Charged Off"]
check = df[df["loan_status"].isin(closed_status)]

for c in ["hardship_flag", "pymnt_plan", "policy_code"]:
    print(c)
    print(pd.crosstab(check[c], check["loan_status"]))
    print()

hardship_flag
loan_status    Charged Off  Fully Paid
hardship_flag                         
N                   268559     1076751

pymnt_plan
loan_status  Charged Off  Fully Paid
pymnt_plan                          
n                 268559     1076751

policy_code
loan_status  Charged Off  Fully Paid
policy_code                         
1.0               268559     1076751



In [14]:
const_cols = ["hardship_flag", "pymnt_plan", "policy_code"]
df = df.drop(columns=const_cols)

In [15]:
check = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]
print(check.nunique()[check.nunique() == 1])

Series([], dtype: int64)


In [16]:
closed = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]
top_share = closed.apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0])
print(top_share[top_share > 0.99].sort_values(ascending=False))

delinq_amnt                 0.996284
acc_now_delinq              0.995289
disbursement_method         0.994871
chargeoff_within_12_mths    0.991901
dtype: float64


In [17]:
near_const = ["delinq_amnt", "acc_now_delinq", "disbursement_method", "chargeoff_within_12_mths"]
df = df.drop(columns=near_const)
print(df.shape[1]) 

87


In [18]:
print(df.select_dtypes("object").nunique())

id                     2260668
term                         2
grade                        7
sub_grade                   35
emp_length                  11
home_ownership               6
verification_status          3
issue_d                    139
loan_status                  9
purpose                     14
zip_code                   956
addr_state                  51
earliest_cr_line           754
initial_list_status          2
application_type             2
dtype: int64


In [ ]:
# term: " 36 months" -> 36
df["term"] = df["term"].str.extract(r"(\d+)").astype(int)

# emp_length
emp_map = {"< 1 year": 0, "1 year": 1, "10+ years": 10}
emp_map.update({f"{i} years": i for i in range(2, 10)})
df["emp_length"] = df["emp_length"].map(emp_map)

# sub_grade: A1 -> 0 ... G5 -> 34
order = [f"{g}{n}" for g in "ABCDEFG" for n in range(1, 6)]
df["sub_grade"] = df["sub_grade"].map({s: i for i, s in enumerate(order)})

issue = pd.to_datetime(df["issue_d"], format="%b-%Y")
first = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y")
df["credit_age_years"] = (issue - first).dt.days / 365
df = df.drop(columns=["earliest_cr_line"])

cat_cols = ["home_ownership", "verification_status", "purpose", "addr_state",
            "zip_code", "initial_list_status", "application_type"]
for c in cat_cols:
    df[c] = df[c].astype("category")

In [20]:
df.dtypes

id                                  object
loan_amnt                          float64
funded_amnt                        float64
funded_amnt_inv                    float64
term                                 int64
int_rate                           float64
installment                        float64
grade                               object
sub_grade                            int64
emp_length                         float64
home_ownership                    category
annual_inc                         float64
verification_status               category
issue_d                             object
loan_status                         object
purpose                           category
zip_code                          category
addr_state                        category
dti                                float64
delinq_2yrs                        float64
fico_range_low                     float64
fico_range_high                    float64
inq_last_6mths                     float64
mths_since_

In [21]:
print(df['loan_status'].value_counts())

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [22]:
closed = {
    "Fully Paid": 0,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Charged Off": 1,
    "Does not meet the credit policy. Status:Charged Off": 1,
}

df_predict = df[~df["loan_status"].isin(closed)].copy()
df = df[df["loan_status"].isin(closed)].copy()
df["target"] = df["loan_status"].map(closed)

print(df.shape)              # (1348059, 88)
print(df_predict.shape)      # (912609, 87)
print(df["target"].mean())   # 0.1998

(1348059, 88)
(912609, 87)
0.19978354063138187


In [23]:
df = df.drop(columns=["grade"])
df_predict = df_predict.drop(columns=["grade"])
print(df.shape[1], df_predict.shape[1]) 

87 86


In [24]:
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df_predict = df_predict.drop(columns=["issue_d"])
print(df["issue_d"].min(), df["issue_d"].max())
print(df_predict.shape[1])   

2007-06-01 00:00:00 2018-12-01 00:00:00
85


In [25]:
print(pd.crosstab(df["issue_d"].dt.year, df["target"]))

target        0      1
issue_d               
2007        445    158
2008       1897    496
2009       4558    723
2010      10780   1757
2011      18424   3297
2012      44723   8644
2013     113780  21024
2014     181941  41161
2015     299742  75803
2016     224853  68242
2017     130152  39148
2018      47444   8867


In [26]:
summary = (
    df.groupby(df["issue_d"].dt.year)["target"]
      .mean()
      .mul(100)
      .round(1)
      .to_frame("charged_off_pct")
)
summary["fully_paid_pct"] = (100 - summary["charged_off_pct"]).round(1)
summary.index.name = "year"

print(summary[["fully_paid_pct", "charged_off_pct"]])

      fully_paid_pct  charged_off_pct
year                                 
2007            73.8             26.2
2008            79.3             20.7
2009            86.3             13.7
2010            86.0             14.0
2011            84.8             15.2
2012            83.8             16.2
2013            84.4             15.6
2014            81.6             18.4
2015            79.8             20.2
2016            76.7             23.3
2017            76.9             23.1
2018            84.3             15.7


In [27]:
import pandas as pd

closed_status = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid",
    "Charged Off",
    "Does not meet the credit policy. Status:Charged Off",
]

orig = pd.read_csv("/kaggle/working/accepted_2007_to_2018Q4.csv", usecols=["id", "loan_status", "issue_d"])
orig["year"] = pd.to_datetime(orig["issue_d"], format="%b-%Y").dt.year
orig["is_closed"] = orig["loan_status"].isin(closed_status)

check = (
    orig.groupby("year")["is_closed"]
        .agg(closed="sum", total="count")
)
check["open"] = check["total"] - check["closed"]
check["closed_pct"] = (check["closed"] / check["total"] * 100).round(1)
check["open_pct"] = (check["open"] / check["total"] * 100).round(1)

print(check[["closed", "open", "total", "closed_pct", "open_pct"]])

/tmp/ipykernel_50/1927545034.py:10: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  orig = pd.read_csv("/kaggle/working/accepted_2007_to_2018Q4.csv", usecols=["id", "loan_status", "issue_d"])


        closed    open   total  closed_pct  open_pct
year                                                
2007.0     603       0     603       100.0       0.0
2008.0    2393       0    2393       100.0       0.0
2009.0    5281       0    5281       100.0       0.0
2010.0   12537       0   12537       100.0       0.0
2011.0   21721       0   21721       100.0       0.0
2012.0   53367       0   53367       100.0       0.0
2013.0  134804      10  134814       100.0       0.0
2014.0  223102   12527  235629        94.7       5.3
2015.0  375545   45550  421095        89.2      10.8
2016.0  293095  141312  434407        67.5      32.5
2017.0  169300  274279  443579        38.2      61.8
2018.0   56311  438931  495242        11.4      88.6


In [28]:
features = [c for c in df.columns if c not in ["id", "loan_status", "issue_d", "target"]]
X_predict = df_predict[features]

In [29]:
features = [c for c in df.columns if c not in ["id", "loan_status", "issue_d", "target"]]

cat_cols = df[features].select_dtypes(include=["object", "category"]).columns.tolist()
print(len(features), "features,", len(cat_cols), "categorical")
print(df[cat_cols].nunique().sort_values(ascending=False))

# same categories in df and df_predict, so the model sees consistent encoding
for c in cat_cols:
    cats = pd.Index(pd.concat([df[c], df_predict[c]]).dropna().unique())
    dtype = pd.CategoricalDtype(cats)
    df[c] = df[c].astype(dtype)
    df_predict[c] = df_predict[c].astype(dtype)

year = df["issue_d"].dt.year
train_mask = year <= 2014
val_mask = year == 2015
test_mask = year == 2016

X_train, y_train = df.loc[train_mask, features], df.loc[train_mask, "target"]
X_val, y_val = df.loc[val_mask, features], df.loc[val_mask, "target"]
X_test, y_test = df.loc[test_mask, features], df.loc[test_mask, "target"]

print("train:", X_train.shape, round(y_train.mean(), 4))
print("val:  ", X_val.shape, round(y_val.mean(), 4))
print("test: ", X_test.shape, round(y_test.mean(), 4))

83 features, 7 categorical
zip_code               945
addr_state              51
purpose                 14
home_ownership           6
verification_status      3
initial_list_status      2
application_type         2
dtype: int64
train: (453808, 83) 0.1702
val:   (375545, 83) 0.2018
test:  (293095, 83) 0.2328


In [32]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

features2 = [c for c in features if c != "zip_code"]
Xtr, Xva, Xte = X_train[features2], X_val[features2], X_test[features2]
print("features used:", len(features2))

model = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.05, num_leaves=63,
    min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, reg_lambda=5,
    n_jobs=-1, random_state=42, verbose=-1,
)
model.fit(Xtr, y_train, eval_set=[(Xva, y_val)], eval_metric="auc",
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])

best_n = model.best_iteration_
print("best n_estimators:", best_n)

p_val = model.predict_proba(Xva)[:, 1]
p_test = model.predict_proba(Xte)[:, 1]

print("\nAUC val 2015: ", round(roc_auc_score(y_val, p_val), 4))
print("AUC test 2016:", round(roc_auc_score(y_test, p_test), 4))
print("\nmean predicted prob (val):", round(p_val.mean(), 4), "| actual rate:", round(y_val.mean(), 4))

features used: 82
best n_estimators: 321

AUC val 2015:  0.7444
AUC test 2016: 0.7229

mean predicted prob (val): 0.1774 | actual rate: 0.2018


In [33]:
import pandas as pd

X_all, y_all = df[features2], df["target"]
print("final train:", X_all.shape, "| default rate:", round(y_all.mean(), 4))

# a bit more data than before, so slightly more trees than the early-stopping result
n_final = int(best_n * 1.1)
print("n_estimators used:", n_final)

final_model = lgb.LGBMClassifier(
    n_estimators=n_final, learning_rate=0.05, num_leaves=63,
    min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, reg_lambda=5,
    n_jobs=-1, random_state=42, verbose=-1,
)
final_model.fit(X_all, y_all)

out = df_predict[["id"]].copy()
out["p_default"] = final_model.predict_proba(df_predict[features2])[:, 1]
out["p_paid"] = 1 - out["p_default"]
out["risk_decile"] = pd.qcut(out["p_default"].rank(method="first"), 10, labels=range(1, 11)).astype(int)

print(out["p_default"].describe())
print(out.groupby("risk_decile")["p_default"].mean().round(3))

out.to_csv("predictions_for_powerbi.csv", index=False)
print("saved:", out.shape)

final train: (1348059, 82) | default rate: 0.1998
n_estimators used: 353
count    912609.000000
mean          0.213671
std           0.139693
min           0.005722
25%           0.102560
50%           0.186633
75%           0.299047
max           0.848724
Name: p_default, dtype: float64
risk_decile
1     0.035
2     0.071
3     0.103
4     0.135
5     0.169
6     0.206
7     0.248
8     0.300
9     0.369
10    0.502
Name: p_default, dtype: float64
saved: (912609, 4)


In [34]:
import pandas as pd

chk = pd.DataFrame({"p": p_test, "y": y_test.values})
chk["decile"] = pd.qcut(chk["p"].rank(method="first"), 10, labels=range(1, 11)).astype(int)

table = chk.groupby("decile").agg(
    loans=("y", "size"),
    predicted=("p", "mean"),
    actual=("y", "mean"),
).round(3)
print(table)

        loans  predicted  actual
decile                          
1       29310      0.029   0.041
2       29309      0.056   0.085
3       29310      0.080   0.120
4       29309      0.104   0.157
5       29310      0.129   0.195
6       29309      0.158   0.228
7       29309      0.192   0.270
8       29310      0.236   0.316
9       29309      0.300   0.387
10      29310      0.435   0.528


In [35]:
print(orig.loc[~orig["is_closed"], "loan_status"].value_counts())

loan_status
Current               878317
Late (31-120 days)     21467
In Grace Period         8436
Late (16-30 days)       4349
Default                   40
Name: count, dtype: int64


In [36]:
out["loan_status"] = df_predict["loan_status"].values

print(out.groupby("loan_status")["p_default"].agg(["count", "mean"]).round(3))

out.to_csv("predictions_for_powerbi.csv", index=False)
print("saved:", out.shape)

                     count   mean
loan_status                      
Current             878317  0.210
Default                 40  0.285
In Grace Period       8436  0.290
Late (16-30 days)     4349  0.294
Late (31-120 days)   21467  0.302
saved: (912609, 5)


In [37]:
dist = pd.crosstab(out["risk_decile"], out["loan_status"], normalize="columns").round(3)
print(dist)

loan_status  Current  Default  In Grace Period  Late (16-30 days)  \
risk_decile                                                         
1              0.103    0.025            0.027              0.022   
2              0.102    0.075            0.042              0.041   
3              0.102    0.025            0.052              0.057   
4              0.101    0.075            0.074              0.070   
5              0.101    0.125            0.090              0.087   
6              0.100    0.075            0.102              0.095   
7              0.099    0.100            0.115              0.112   
8              0.098    0.150            0.132              0.134   
9              0.097    0.100            0.157              0.159   
10             0.095    0.250            0.209              0.222   

loan_status  Late (31-120 days)  
risk_decile                      
1                         0.017  
2                         0.035  
3                         0.050  
4